# P10 원본 재현 감사 1/2 — T4

본평가·학습은 완료되어 다시 선택하지 않습니다. 이 노트북은 고정 원본과 결과의 **재현 감사**입니다. 반환 감사 전에는 P10 완료가 아닙니다.

Drive `boolean_interp_v1_4/`에 기존 `v1_4_p10_update_evaluation_r1.zip`과 새 `v1_4_p10_source_audit_r1.zip`을 둡니다. 기존 `P10_update_evaluation_r1` 전체 cache/arrays/latents/예측/fit 결과를 유지하세요. 새 출력은 `P10_update_source_audit_r1`입니다.

T4에서 trained/init 6개 모델의 예약 Update 80k 위치 전체를 재추출하여 54개 원본 배열과 비교합니다. 이어 선택 dictionary 12개의 모든 latent·fidelity·train 통계를 독립 계산합니다. 완료 후 GPU 연결을 종료하고 2/2 CPU 노트북으로 이동하세요.

호출당 기본 2시간이며 chunk/task 경계에서 PAUSED이면 같은 실행 셀을 반복합니다. 런타임이 끊기면 해당 노트북 위부터 실행합니다. 과제 수·budget·허용오차를 바꾸지 마세요. 완료 또는 중단 시 마지막 셀의 ZIP·checksum을 반환합니다.


In [ ]:
from google.colab import drive,files
from pathlib import Path
import hashlib,json,zipfile,sys,subprocess,os,time,shutil
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/boolean_interp_v1_4')
ROOT=Path('/content/MI_P10_source_audit_r1');ROOT.mkdir(exist_ok=True)
def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
for filename,expected in [('v1_4_p10_update_evaluation_r1.zip','69bb4191321ed5228cdb5a5dd0e90f9cbeaf1e506aedd1a85a87c9ef87b22e1e'),('v1_4_p10_source_audit_r1.zip','3e1ca7515cd4048e2a7cf9633476a91f9b87d735f83628dc679cfa46d2a39489')]:
    bundle=BASE/filename
    assert bundle.exists(), f'필요한 입력 파일: {bundle}'
    assert digest(bundle)==expected, 'ZIP checksum 불일치'
    with zipfile.ZipFile(bundle) as z:
        for n in z.namelist():assert (ROOT/n).resolve().is_relative_to(ROOT.resolve())
        z.extractall(ROOT)
SOURCE=BASE/'P10_update_evaluation_r1'
OUTPUT=BASE/'P10_update_source_audit_r1';OUTPUT.mkdir(exist_ok=True)

In [ ]:
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUTPUT/f'install_{time.time_ns()}.txt').write_text(log.stdout);print(log.stdout);log.check_returncode()
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2';os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
sys.path.insert(0,str(ROOT))
from interp_v1_4.p10_source_audit import verify,export
verify(ROOT)
def run(args):
    path=OUTPUT/'logs'/f'{time.time_ns()}.txt';path.parent.mkdir(exist_ok=True)
    with path.open('w') as log:
        p=subprocess.Popen([sys.executable,'-u']+args,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in p.stdout:print(line,end='');log.write(line);log.flush()
            rc=p.wait()
        except BaseException:p.terminate();p.wait();raise
    assert rc==0, f'실행 실패: {path}; 마지막 셀에서 증빙을 반환하세요'
run(['-m','pytest','tests_v1_4/test_p10_source_audit.py','-q'])
MAX_SECONDS=7200  # 호출별 chunk/task 경계에서 정지; 중단 후 같은 셀 재실행

In [ ]:
run(['-m','interp_v1_4.p10_source_audit','gpu','--root',str(ROOT),'--source',str(SOURCE),'--output',str(OUTPUT),'--max-seconds',str(MAX_SECONDS)])

In [ ]:
print('GPU 감사 완료: GPU 연결을 종료하고 CPU 감사로 이동하세요.' if (OUTPUT/'gpu_complete.json').exists() else 'PAUSED: 위 GPU 실행 셀을 반복하세요. CPU 감사는 아직 시작하지 않습니다.')

중간 정지는 실패나 완료가 아닙니다. 완료된 감사 영수증은 재사용하고 남은 과제부터 재개합니다. 마지막 셀에서 감사 ZIP과 checksum을 내려받아 반환하세요.

In [ ]:
archive=Path(f'/content/v1_4_p10_source_audit_return_{time.time_ns()}.zip')
export(OUTPUT,archive)
back=OUTPUT/'exports';back.mkdir(exist_ok=True)
shutil.copyfile(archive,back/archive.name);shutil.copyfile(archive.with_suffix('.sha256.json'),back/archive.with_suffix('.sha256.json').name)
print('checksum',digest(archive));files.download(str(archive));files.download(str(archive.with_suffix('.sha256.json')))